# 04 — Eventos candidatos sin encadenamiento ilimitado

Una fila FIRMS es una **detección**, no un incendio. Un incendio puede producir varias filas por píxeles cercanos, días consecutivos y distintos satélites.

Aquí agrupamos detecciones cercanas para crear **eventos candidatos**. El algoritmo usa:

- distancia de enlace: 1,5 km;
- separación temporal máxima: 2 días;
- duración total máxima: 3 días inclusivos;
- diagonal espacial máxima del grupo: 3 km.

Los dos últimos límites corrigen el efecto de “cadena”: A puede estar cerca de B y B de C, pero eso ya no permite formar grupos arbitrariamente largos.

**Dos niveles de agrupación.** El tope de 3 km puede partir un incendio grande en varios eventos. Por eso se calcula también un nivel grueso, el **complejo** (hasta 10 km y 14 días, mismo enlace de 1,5 km y 2 días). Son **dos escenarios de agrupamiento**, no cotas del número real de incendios: un complejo puede unir incendios distintos y un incendio grande puede seguir dividido en varios eventos. La diferencia entre ambos escenarios se calcula abajo.

> Estos eventos son una herramienta analítica. No son partes oficiales de incendio ni miden superficie quemada.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image

ROOT = Path.cwd()
SERIES = ROOT / "DATOS_PROCESADOS" / "series"
FIGURES = ROOT / "DATOS_PROCESADOS" / "figuras"

In [2]:
from pipeline_tunari_avanzado import construir_eventos, construir_metricas, validar_eventos_documentados

datos = pd.read_parquet(ROOT / 'DATOS_PROCESADOS/zona_tunari_limite_referencia_utm.parquet')
eventos_det, catalogo = construir_eventos(datos)
metricas = construir_metricas(eventos_det)
validacion = validar_eventos_documentados(eventos_det)

print(f"Detecciones: {len(eventos_det):,}")
n_complejos = eventos_det['complejo_id'].nunique()
print(f"Eventos candidatos (escenario fino): {len(catalogo):,}")
print(f"Complejos candidatos (escenario grueso): {n_complejos:,} ({100 * (1 - n_complejos / len(catalogo)):.1f} % menos que los eventos)")
print(f"Duración máxima observada: {catalogo['duracion_dias_inclusivos'].max()} días")
print(f"Diagonal máxima observada: {catalogo['diagonal_bbox_km'].max():.3f} km")
display(catalogo.head(10))

Detecciones: 5,646
Eventos candidatos (escenario fino): 1,944
Complejos candidatos (escenario grueso): 1,872 (3.7 % menos que los eventos)
Duración máxima observada: 3 días
Diagonal máxima observada: 2.996 km


,event_num,event_id,fecha_inicio,fecha_fin,detecciones,celdas_1km,x_min_m,x_max_m,y_min_m,y_max_m,frp_suma_mw,frp_max_mw,plataformas,fuentes,duracion_dias_inclusivos,diagonal_bbox_km,es_multisensor
0,1,TUNARI-00001,2001-08-09,2001-08-09,2,2,162774.851554,163836.261132,8.082352e+06,8.082524e+06,29.3,18.7,Terra,MODIS-Terra,1,1.075352,False
1,2,TUNARI-00002,2001-08-14,2001-08-14,1,1,139703.712598,139703.712598,8.065866e+06,8.065866e+06,12.8,12.8,Terra,MODIS-Terra,1,0.000000,False
2,3,TUNARI-00003,2001-09-19,2001-09-19,2,2,158411.059519,159403.283620,8.084778e+06,8.084916e+06,57.8,43.7,Terra,MODIS-Terra,1,1.001856,False
3,4,TUNARI-00004,2002-09-23,2002-09-23,2,2,118293.741422,119292.167120,8.098163e+06,8.098337e+06,25.1,17.0,Aqua,MODIS-Aqua,1,1.013412,False
4,5,TUNARI-00005,2003-06-01,2003-06-01,2,2,159958.347485,160364.426454,8.116376e+06,8.117624e+06,67.0,47.0,Terra,MODIS-Terra,1,1.311984,False
5,6,TUNARI-00006,2003-06-20,2003-06-20,1,1,167494.542587,167494.542587,8.113063e+06,8.113063e+06,9.0,9.0,Aqua,MODIS-Aqua,1,0.000000,False
6,7,TUNARI-00007,2003-06-22,2003-06-22,3,3,186416.281103,188411.813440,8.084667e+06,8.084996e+06,51.6,21.5,Aqua,MODIS-Aqua,1,2.022565,False
7,8,TUNARI-00008,2003-06-24,2003-06-24,1,1,111814.593946,111814.593946,8.093652e+06,8.093652e+06,7.4,7.4,Aqua,MODIS-Aqua,1,0.000000,False
8,9,TUNARI-00009,2003-06-24,2003-06-24,1,1,168329.466838,168329.466838,8.108833e+06,8.108833e+06,12.7,12.7,Aqua,MODIS-Aqua,1,0.000000,False
9,10,TUNARI-00010,2003-07-01,2003-07-01,1,1,186143.187005,186143.187005,8.087210e+06,8.087210e+06,6.8,6.8,Aqua,MODIS-Aqua,1,0.000000,False


## Sensibilidad de parámetros

No existe una única distancia “verdadera” sin información de campo. Por eso repetimos el agrupamiento con varias combinaciones y guardamos cuántos eventos produce cada una. Si el resultado cambia mucho, esa incertidumbre debe acompañar cualquier conclusión.

In [3]:
sens = pd.read_csv(SERIES / 'tunari_sensibilidad_eventos_acotados.csv')
display(sens.pivot(index='distancia_km', columns='separacion_max_dias', values='eventos'))

separacion_max_dias,1,2,3
distancia_km,,,
1.0,2205,2186,2175
1.5,1972,1944,1934
2.0,1898,1863,1850


## Validación con casos documentados (con línea base)

Comparamos siete fechas publicadas sobre incendios del Tunari con FIRMS en ventanas de 0, ±1 y ±2 días.

**Cuidado con la coincidencia simple.** En julio–octubre casi cualquier día tiene alguna detección cerca: con ±2 días, el 93 % de los días de esa temporada (2019–2025) tiene al menos una. Que un caso “coincida” no demuestra nada por sí solo. Por eso se reportan **dos percentiles**, que responden preguntas distintas:

- frente a **todos los días de la temporada** (julio–octubre de 2019–2025): ¿el caso cae en un periodo de mucho fuego?
- frente a **los días de su propio mes y año**: ¿FIRMS distingue ese día dentro de su mes? Es más exigente y castiga los casos que ocurren en un mes extremo.

Un caso destaca si supera el percentil 75 (umbral arbitrario, solo de referencia). Con solo 7 casos, ambos criterios son frágiles.

Además, las noticias no incluyen coordenadas ni perímetros, así que no se puede validar la ubicación.

In [4]:
v2 = validacion[validacion['ventana_mas_menos_dias'].eq(2)]
print(f"Probabilidad de que un día cualquiera de la temporada tenga detección en ±2 días: {v2['prob_dia_azar_con_deteccion'].iloc[0]:.1%}")
print(f"Casos con detección en ±2 días: {int(v2['coincidencia_temporal'].sum())} de {len(v2)}")
print(f"Casos que superan el percentil 75 de la temporada: {int(v2['destaca_sobre_temporada_p75'].sum())} de {len(v2)}")
print(f"Casos que superan el percentil 75 de su mismo mes y año: {int(v2['destaca_mismo_mes_anio_p75'].sum())} de {len(v2)}")
display(v2[['fecha_documentada','lugar_reportado','detecciones_firms','complejos_candidatos','percentil_en_temporada','percentil_mismo_mes_anio','destaca_sobre_temporada_p75','destaca_mismo_mes_anio_p75']].round(2))

Probabilidad de que un día cualquiera de la temporada tenga detección en ±2 días: 93.4%
Casos con detección en ±2 días: 7 de 7
Casos que superan el percentil 75 de la temporada: 5 de 7
Casos que superan el percentil 75 de su mismo mes y año: 4 de 7


,fecha_documentada,lugar_reportado,detecciones_firms,complejos_candidatos,percentil_en_temporada,percentil_mismo_mes_anio,destaca_sobre_temporada_p75,destaca_mismo_mes_anio_p75
14,2019-09-29,Tirani/Tirane,30,9,0.84,0.98,True,True
15,2020-07-26,Tirani,6,5,0.39,0.06,False,False
16,2020-08-02,Sacaba/Tiquipaya,122,31,0.99,0.82,True,True
17,2020-08-16,Taquiña,3,3,0.25,0.15,False,False
18,2020-08-27,Candelaria Oeste,53,13,0.95,0.63,True,False
19,2024-09-10,Molinos/Tiquipaya,46,14,0.93,0.88,True,True
20,2025-08-19,Tirani,59,4,0.96,0.87,True,True


Los casos pequeños (por ejemplo, el de unas 3 ha) no destacan: es esperable, y también indica que FIRMS no ve todos los fuegos. El umbral del percentil 75 es arbitrario y sirve solo como referencia.